In [5]:
import pandas as pd

# Cargamos los datasets exportados desde SQLite.
# Cada archivo representa una parte distinta del sistema:
# - profiles: perfiles detectados por sesión
# - responses: respuestas textuales del usuario
# - sessions: metadatos de cada sesión (timestamps, estado final)
profiles = pd.read_csv("../data/profiles_38_52.csv")
responses = pd.read_csv("../data/responses_172_244.csv")
sessions = pd.read_csv("../data/sessions_56_75.csv")

profiles.info()
responses.info()
sessions.info()


profiles.head()
responses.head()
sessions.head()
# Extraemos la categoría principal del perfil.
# El formato es 'categoria_indice' (ej: 'tec_0'), así que nos quedamos con lo anterior al '_'.
profiles["categoria"] = profiles["perfil_id"].str.split("_").str[0]

# Contamos cuántas veces aparece cada categoría vocacional.
# Esto nos da una primera idea de qué perfiles son más comunes en el conjunto analizado.
profiles["categoria"].value_counts()

len(sessions)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          15 non-null     int64  
 1   session_id  15 non-null     object 
 2   perfil_id   15 non-null     object 
 3   score       15 non-null     float64
 4   created_at  15 non-null     int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 732.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          73 non-null     int64 
 1   session_id  73 non-null     object
 2   texto       73 non-null     object
 3   created_at  73 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 2.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------ 

9

In [1]:
import json
import pandas as pd

# 1. Cargar datos (ejecuta esto primero)
profiles = pd.read_csv("../data/profiles_38_52.csv")
responses = pd.read_csv("../data/responses_172_244.csv")
sessions = pd.read_csv("../data/sessions_56_75.csv")

# 2. Filtrar sesiones con estado
sessions_con_estado = sessions.dropna(subset=["state"])

# 3. Extraer perfil + recomendaciones
filas = []

for s in sessions_con_estado["state"]:
    data = json.loads(s)  # <-- AQUÍ ESTABA EL ERROR
    perfil = data.get("categoriaPrincipal")
    recomendaciones = data.get("respuestasVocacionales", [])
    
    for rec in recomendaciones:
        filas.append({"perfil": perfil, "recomendacion": rec})

df_rec = pd.DataFrame(filas)

# 4. Agrupar por perfil y recomendación
frecuencias = df_rec.groupby(["perfil", "recomendacion"]).size().reset_index(name="frecuencia")
frecuencias


,perfil,recomendacion,frecuencia
0,social,"Disfruto mucho ayudando a la gente, enseñando ...",1
1,social,Docencia y formación,1
2,tecnica,Programar,1
3,tecnica,Sofware,1


In [18]:
for s in sessions_con_estado["state"]:
    try:
        data = json.loads(s)
    except:
        continue
    
    perfil = (
        data.get("perfil") or 
        data.get("categoriaPrincipal") or 
        data.get("categoria") or 
        None
    )
    
    if perfil:
        print("Perfil encontrado:", perfil)
        print(data)
        print("---------------")


Perfil encontrado: tecnica
{'respuestasVocacionales': ['Programar', 'Sofware'], 'respuestasModificadores': ['Flexibilidad', 'Autodidacta'], 'categoriaPrincipal': 'tecnica', 'modificadorEntorno': 'flexible', 'modificadorAprendizaje': 'autodidacta'}
---------------
Perfil encontrado: social
{'respuestasVocacionales': ['Disfruto mucho ayudando a la gente, enseñando cosas nuevas y apoyando a los estudiantes con dificultades en sus estudios.', 'Docencia y formación'], 'respuestasModificadores': ['Flexibilidad.', 'Autodidacta.'], 'categoriaPrincipal': 'social', 'modificadorEntorno': None}
---------------
